# Forward Selection Across Every Variable

Applies the variable-selection procedure from **BA222 Lecture Notes 08** (Rhoads,
*Prediction in Multivariate Regression*) to this project's ad-set x day panel, over
**every** variable in the dataset rather than only the eight declared drivers.

LN08 Section 3.1, verbatim:

1. Compute the correlation with the dependent variable for all variables in the dataset.
2. The first independent variable is the one with the highest correlation coefficient.
3. Add new variables according to their contribution to adjusted R-squared: estimate
   models that keep the variables already chosen and add one more, take the best.
4. Repeat step 3 starting from the model chosen at the end of it.
5. Stop once no model increases adjusted R-squared.

The variables left out at the end are what the notes call **redundant variables** --
not variables without predictive power, but variables whose predictive power is already
covered by what is in the model.

Section 3.2's **backward selection** (drop the least significant variable until every
survivor clears p < 0.05) is implemented at the end too, because LN08 recommends running
both and keeping whichever gives the higher adjusted R-squared.

**Run this from the project root**, not from a subdirectory -- the data path below is
relative.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)

## 1. Load the panel

`Dataset/master_dataset.xlsx` is the curated all-variable table: one row per ad set per
day, every collected variable in one place. See `Vault/Data Pipeline/Master-Dataset.md`
for how it is built.

Two things to know about this file before reading any result off it:

- **It ends 2026-07-31 and covers 18 ad sets.** The live application database has since
  grown past it. Rebuild with `python build_master_dataset.py` to refresh.
- **The change-related columns here are detector-inferred, not recorded.**
  `ad_set_change_type`, `ad_change_type`, `ad_set_change_recency`, `ad_change_recency`
  and the `*_today` flags came from step-shift/activity heuristics. Those heuristics were
  removed from the application on 2026-08-06 after they were found to be wrong. Anything
  this notebook concludes about them is provisional.

In [ ]:
DATA_FILE = Path("Dataset/master_dataset.xlsx")
SHEET = "master_adset_daily"
TARGET = "leads"

raw = pd.read_excel(DATA_FILE, sheet_name=SHEET)
raw["date"] = pd.to_datetime(raw["date"])

print(f"{len(raw):,} rows | {raw['ad_set_id'].nunique()} ad sets | "
      f"{raw['date'].min():%Y-%m-%d} to {raw['date'].max():%Y-%m-%d}")
print(f"{len(raw.columns)} columns")
raw.head()

## 2. What is excluded from the candidate pool, and why

Everything not listed here is a candidate. Four kinds of exclusion:

- **Identifiers** -- a row label is not a driver.
- **Target leakage** -- a variable computed from `leads`, or a second measurement of the
  same event. A model that uses one of these will look excellent and forecast nothing,
  because the value is not known before the leads are.
- **Zero variance** -- a constant column has no correlation with anything.
- **Duplicate encodings** -- the same information in two columns. Keeping both leaves the
  design matrix collinear and splits one signal across two coefficients.

Every mechanical claim below is re-checked against the data in the next cell rather than
taken on trust, so a future refresh that breaks one of them fails loudly here instead of
quietly changing the results.

In [ ]:
EXCLUDED = {
    "date": "identifier / time index",
    "ad_set_id": "identifier",
    "campaign_id": "identifier",
    "campaign_name": "identifier (16 levels; see CAMPAIGN_FIXED_EFFECTS below)",
    "cpl": "leakage: exactly spend / leads, and undefined on zero-lead days",
    "leads_new": "leakage: a component of the target (leads_new <= leads on every row)",
    "leads_meta_reported": "leakage: Meta's own count of the event the target measures",
    "holiday_name": "text label, missing on 864 of 879 rows; is_holiday carries it",
    "ad_set_budget_type": "constant in this window (zero variance)",
    "day_of_week_num": "1:1 with day_of_week, which enters as a categorical block",
    "is_weekend": "a coarsening of day_of_week, already a candidate",
    "holiday_proximity": "1:1 with is_holiday in this window (only 2 of 5 buckets occur)",
}

checks = {
    "cpl == spend / leads": np.allclose(
        raw["cpl"].dropna(),
        (raw["spend"] / raw["leads"].replace(0, np.nan)).dropna(),
    ),
    "leads_new <= leads on every row": bool((raw["leads_new"] <= raw["leads"]).all()),
    "ad_set_budget_type is constant": raw["ad_set_budget_type"].nunique() == 1,
    "day_of_week_num is 1:1 with day_of_week":
        raw.groupby("day_of_week")["day_of_week_num"].nunique().eq(1).all(),
    "holiday_proximity is 1:1 with is_holiday":
        raw.groupby("holiday_proximity")["is_holiday"].nunique().eq(1).all()
        and raw["holiday_proximity"].nunique() == raw["is_holiday"].nunique(),
    "is_weekend is implied by day_of_week":
        raw.groupby("day_of_week")["is_weekend"].nunique().eq(1).all(),
}

for claim, holds in checks.items():
    print(f"{'OK  ' if holds else 'FAIL'}  {claim}")
assert all(checks.values()), "an exclusion no longer matches the data -- revisit EXCLUDED"

## 3. Build the candidate pool

Two switches:

- `CANDIDATE_SET` -- `"all"` uses every non-excluded column, which is what this notebook
  is for. `"declared"` restricts to the eight declared drivers, which reproduces the
  framework the application's own OLS panel is limited to, for comparison.
- `CAMPAIGN_FIXED_EFFECTS` -- adds `campaign_name` as a 16-level block. It will lift
  adjusted R-squared, because it lets the model learn a separate baseline per campaign.
  That is a control, not a driver: it explains *that* campaigns differ, never *why*.

**Categorical variables enter as whole blocks** (`C(day_of_week)` is one candidate, not
seven). LN08 does the same in Section 3.2 -- it declines to drop `C(buildingStyle)` while
any of its levels is significant -- and it keeps a result readable as a statement about a
variable rather than about Wednesdays.

Rows with a missing value in any candidate are dropped **once, up front**. Adjusted
R-squared is only comparable between models fitted on the same rows, and statsmodels
drops missing rows per model, so letting it do that silently would make every comparison
in this notebook slightly dishonest.

In [ ]:
CANDIDATE_SET = "all"          # "all" | "declared"
CAMPAIGN_FIXED_EFFECTS = False

DECLARED_DRIVERS = [
    "spend",                    # 2
    "is_holiday",               # 3 (holiday_proximity's surviving encoding)
    "days_since_adset_started", # 4
    "frequency",                # 5
    "ad_change_recency",        # 6
    "ad_set_change_recency",    # 7
    "day_of_week",              # 8
]

pool = [c for c in raw.columns if c != TARGET and c not in EXCLUDED]
if CANDIDATE_SET == "declared":
    pool = [c for c in DECLARED_DRIVERS if c in pool]
if CAMPAIGN_FIXED_EFFECTS:
    pool = pool + ["campaign_name"]

analysis = raw[[TARGET] + pool].copy()
dropped = len(analysis) - len(analysis.dropna())
analysis = analysis.dropna().reset_index(drop=True)

constant = [c for c in pool if analysis[c].nunique() <= 1]
if constant:
    print(f"dropping {len(constant)} constant column(s) after the row filter: {constant}")
pool = [c for c in pool if c not in constant]

CATEGORICAL = [c for c in pool if analysis[c].dtype == object]

print(f"{len(pool)} candidate variables ({len(CATEGORICAL)} categorical), "
      f"{len(analysis):,} rows ({dropped} dropped for missing values)")
print()
print("categorical:", CATEGORICAL)
print("numeric:    ", [c for c in pool if c not in CATEGORICAL])

## 4. Fitting helpers

One formula builder, one fitter. Everything below goes through these, so a candidate is
always encoded the same way whether it is being screened, added, or removed.

In [ ]:
def term(name: str) -> str:
    """A candidate's formula term: categoricals get C(), which makes the block one unit."""
    return f"C({name})" if name in CATEGORICAL else name


def formula(names) -> str:
    right = " + ".join(term(n) for n in names) if len(names) else "1"
    return f"{TARGET} ~ {right}"


def fit(names):
    return smf.ols(formula(names), data=analysis).fit()


baseline = fit([])
print(formula([]), "->", f"adj R2 = {baseline.rsquared_adj:.4f}")
print("(the intercept-only model: predict the average day, every day)")

## 5. Steps 1-2 -- rank every variable on its own

LN08 seeds the model with the highest correlation coefficient. A Pearson correlation is
undefined for a categorical block, so the ranking below uses the **R-squared of each
variable's own simple regression**, which for a numeric variable is exactly the square of
its correlation -- identical ranking, and it extends to the blocks. The correlation is
printed alongside where it exists, so the LN08 step is still visible.

This table is also the answer to "how much does each variable explain on its own", which
is worth reading on its own terms before any selection happens.

In [ ]:
screen = []
for name in pool:
    model = fit([name])
    screen.append({
        "variable": name,
        "kind": "categorical" if name in CATEGORICAL else "numeric",
        "terms": int(model.df_model),
        "correlation": (np.nan if name in CATEGORICAL
                        else analysis[name].corr(analysis[TARGET])),
        "r_squared": model.rsquared,
        "adj_r_squared": model.rsquared_adj,
        "p_value": model.f_pvalue,
    })

screen = (pd.DataFrame(screen)
          .sort_values("r_squared", ascending=False)
          .reset_index(drop=True))
screen.style.format({
    "correlation": "{:+.4f}", "r_squared": "{:.4f}",
    "adj_r_squared": "{:.4f}", "p_value": "{:.3g}",
}, na_rep="--")

## 6. Steps 3-5 -- greedy forward selection on adjusted R-squared

One deliberate departure from the loop printed in LN08 Section 4: that loop seeds
`maxR2` with the first variable's **R-squared** and then compares later candidates'
**adjusted** R-squared against it. Those are two different quantities, and since adjusted
R-squared is always the smaller of the two, the comparison makes the second variable look
worse than it is and can stop the search one step early. Here the seed model's adjusted
R-squared is recomputed before the loop starts, so every comparison is like-for-like.

Everything else follows Section 3.1 exactly, including the stopping rule: the search ends
the first time no remaining candidate improves adjusted R-squared at all.

`min_gain` defaults to `0.0`, which is LN08's rule as written -- any improvement counts.
Raising it (`min_gain=0.001`) asks for improvements big enough to care about, which is
worth doing once you have seen the path: a variable that earns +0.00002 has passed the
letter of the rule while telling you nothing.

In [ ]:
def forward_selection(candidates, min_gain=0.0, verbose=True):
    """LN08 3.1: seed on the strongest single variable, then add by adjusted R-squared."""
    remaining = list(candidates)
    if not remaining:
        return [], pd.DataFrame()

    # Steps 1-2: the strongest variable on its own.
    seed = max(remaining, key=lambda name: fit([name]).rsquared)
    included = [seed]
    remaining.remove(seed)
    best = fit(included).rsquared_adj
    path = [{"step": 1, "added": seed, "adj_r_squared": best,
             "gain": best - baseline.rsquared_adj, "terms": int(fit(included).df_model)}]
    if verbose:
        print(f"1. {seed:<28} adj R2 = {best:.4f}")

    # Steps 3-5: add the best remaining candidate until none of them helps.
    while remaining:
        scores = {name: fit(included + [name]).rsquared_adj for name in remaining}
        winner = max(scores, key=scores.get)
        if scores[winner] - best <= min_gain:
            if verbose:
                print(f"\nstop: the best remaining candidate ({winner}) would move "
                      f"adj R2 to {scores[winner]:.6f}, a gain of "
                      f"{scores[winner] - best:+.6f}.")
            break
        gain = scores[winner] - best
        best = scores[winner]
        included.append(winner)
        remaining.remove(winner)
        path.append({"step": len(included), "added": winner, "adj_r_squared": best,
                     "gain": gain, "terms": int(fit(included).df_model)})
        if verbose:
            print(f"{len(included)}. {winner:<28} adj R2 = {best:.4f}  (+{gain:.4f})")

    return included, pd.DataFrame(path)


selected, path = forward_selection(pool)          # LN08 as written
print()
print(f"selected {len(selected)} of {len(pool)} variables")
path

In [ ]:
# The same search with a materiality floor, for contrast. Whatever survives here is the
# part of the model that is doing visible work; whatever the run above added beyond this
# is technically an improvement and practically a rounding error.
material, material_path = forward_selection(pool, min_gain=0.001, verbose=False)
print(f"min_gain = 0.001 keeps {len(material)} of {len(pool)}: {material}")
print(f"adj R2 {fit(material).rsquared_adj:.4f}  "
      f"vs {fit(selected).rsquared_adj:.4f} for the {len(selected)}-variable model")
material_path

## 7. The variables that were left out

LN08's "redundant variables". Each one is refitted **against the final model** and shown
with the adjusted R-squared it would produce if added -- so a rejection is always a
number, never a silence. A near-zero negative margin means the variable is genuinely
redundant given what was selected; a large negative one means it actively costs more
degrees of freedom than it earns.

Note that this is a statement about *this* model, not about the variable in isolation:
a variable can rank high in the section 5 table and still be rejected here because
something already selected explains the same variation. That is the whole point of
LN08's size-vs-bedrooms example.

In [ ]:
final_adj = fit(selected).rsquared_adj
rejected = []
for name in [c for c in pool if c not in selected]:
    model = fit(selected + [name])
    rejected.append({
        "variable": name,
        "adj_r_squared_if_added": model.rsquared_adj,
        "margin": model.rsquared_adj - final_adj,
        "own_r_squared": float(screen.loc[screen["variable"] == name, "r_squared"].iloc[0]),
    })

rejected = (pd.DataFrame(rejected)
            .sort_values("margin", ascending=False)
            .reset_index(drop=True))
print(f"final model adj R2 = {final_adj:.4f}")
rejected

## 8. The selected model

In [ ]:
final_model = fit(selected)
print(formula(selected))
print()
print(final_model.summary())

## 9. Backward selection, for comparison (LN08 Section 3.2)

Start with every candidate in the model, drop the least significant variable, refit, and
repeat until everything left clears p < 0.05.

For a categorical block the relevant test is the **joint** one across all its levels, not
the per-level t-tests -- LN08 makes this point when it refuses to drop `C(buildingStyle)`
over one insignificant category. `anova_lm(typ=2)` gives exactly that: one F-test per
variable, blocks included.

LN08 notes the two methods often agree but need not, and says to keep whichever model has
the higher adjusted R-squared.

In [ ]:
def backward_selection(candidates, alpha=0.05, verbose=True):
    """LN08 3.2: drop the least significant variable until all survivors clear alpha."""
    included = list(candidates)
    while len(included) > 1:
        model = fit(included)
        table = anova_lm(model, typ=2).drop(index="Residual")
        # anova_lm labels rows by formula term, e.g. "C(day_of_week)" -- map back.
        p_values = {name: table.loc[term(name), "PR(>F)"] for name in included}
        worst = max(p_values, key=p_values.get)
        if p_values[worst] <= alpha:
            break
        if verbose:
            print(f"drop {worst:<28} p = {p_values[worst]:.3f}")
        included.remove(worst)
    return included


kept = backward_selection(pool)
print()
comparison = pd.DataFrame([
    {"method": "forward (adj R2)", "variables": len(selected),
     "terms": int(fit(selected).df_model), "adj_r_squared": fit(selected).rsquared_adj},
    {"method": "backward (p < 0.05)", "variables": len(kept),
     "terms": int(fit(kept).df_model), "adj_r_squared": fit(kept).rsquared_adj},
])
print("forward :", sorted(selected))
print("backward:", sorted(kept))
print("only forward :", sorted(set(selected) - set(kept)))
print("only backward:", sorted(set(kept) - set(selected)))
print()
comparison

## 10. What this can and cannot tell you

- **Greedy, not exhaustive.** Forward selection walks one path through the space of
  models; it does not test every combination and is not guaranteed to find the best one.
  LN08's own size-vs-bedrooms example is a demonstration that the order variables are
  considered in changes the answer.
- **Selection is not causation.** A high adjusted R-squared says the variable helps
  predict lead volume in this data. It does not say that moving it moves leads. Funnel
  metrics like `impressions`, `reach` and `messaging_conversations` sit downstream of the
  same ad delivery that produces leads -- they co-move with the target without being
  levers anyone can pull. Set `CANDIDATE_SET = "declared"` and re-run to see the model
  restricted to variables that are actually decisions.
- **The rows are not independent.** 879 ad-set days come from 18 ad sets, and days within
  an ad set are correlated. The adjusted R-squared ranking that drives selection is fine,
  but the standard errors and p-values in the summary above are optimistic -- treat them
  as a rough guide, not as inference.
- **`days_since_adset_started` is left-censored on 799 of 879 rows**, meaning the true
  start date was unknown and estimated. The application removed that estimate on
  2026-08-06 for being wrong.
- **The change variables here are inferred, not recorded** (see the note in section 1).
- **Re-run this whenever the data grows.** The greedy path can change with more rows, so
  the selected list is a finding about the current dataset, not a permanent conclusion.